# 03 — Synthetic Gap Generation & Interpolation

**Goal:** Evaluate two interpolation methods (Cubic Spline and B-Spline) for reconstructing corrupted PPG segments.

Approach:
1. **Create synthetic gaps** — punch artificial holes into clean PPG windows (simulating real corruption)
2. **Interpolate** — fill the holes with each method
3. **Score** — compare each method against the known ground truth using RMSE and a custom Imputation Score

Two gap-size distributions are tested:
- **Uniform** — gap sizes drawn uniformly from 2–100 samples
- **Realistic** — gap sizes drawn from the empirical distribution observed in real corrupted data

**Inputs:** `data/processed/training_windows.pkl`

**Outputs:** `data/processed/interpolation_results.pkl`

---
### Pipeline position
```
01 Import → 02 Signal Processing → [03 Interpolation] → 04 ML Data Loading → 05 Features → 06 Models
```

In [ ]:
import pickle
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from scipy.interpolate import splrep, splev, CubicSpline
from sklearn.metrics import mean_squared_error

In [ ]:
# ── Configuration ──────────────────────────────────────────────────────────────
REPO_ROOT     = Path.cwd().parent if Path.cwd().name == 'improved' else Path.cwd().parent.parent
PROCESSED_DIR = REPO_ROOT / 'data' / 'processed'

# Holes per window: up to MAX_HOLES_PER_WINDOW cycles are modified
MAX_HOLES_PER_WINDOW = 4

# Uniform distribution gap size range
UNIFORM_MIN_GAP = 2
UNIFORM_MAX_GAP = 100

# Middle 700-sample region of each 1200-sample window used for cycle detection
# (avoids filter edge effects at the start/end of the window)
CYCLE_REGION_START = 250
CYCLE_REGION_END   = 950

In [ ]:
with open(PROCESSED_DIR / 'training_windows.pkl', 'rb') as f:
    training_windows = pickle.load(f)

print(f'Loaded {len(training_windows)} training windows')

## 1. Cardiac Cycle Detection

Cycles are detected in the middle 700 samples of each window (samples 250–950) to avoid Butterworth filter edge effects at the boundaries.

A valid cycle is a min → max → min triplet with exactly one systolic peak between two diastolic troughs.

In [ ]:
def find_cycles(window_values: np.ndarray,
                region_start: int = CYCLE_REGION_START,
                region_end:   int = CYCLE_REGION_END) -> list:
    """
    Find valid cardiac cycles in the middle region of a PPG window.

    Returns a list of (min_start, max_peak, min_end) index tuples,
    where indices are relative to the full window (not just the region).
    """
    region = window_values[region_start:region_end]

    # Identify local minima and maxima using sign changes of the first difference
    diff = np.diff(region)
    maxima = np.where((diff[:-1] > 0) & (diff[1:] <= 0))[0] + 1
    minima = np.where((diff[:-1] < 0) & (diff[1:] >= 0))[0] + 1

    cycles = []
    for i in range(len(minima) - 1):
        min_start = minima[i]
        min_end   = minima[i + 1]
        peaks_between = maxima[(maxima > min_start) & (maxima < min_end)]
        if len(peaks_between) == 1:
            # Convert region-relative indices back to full-window indices
            cycles.append((
                min_start + region_start,
                peaks_between[0] + region_start,
                min_end + region_start,
            ))
    return cycles


# Quick sanity check
min_cycles = min(len(find_cycles(df['PLETH'].values)) for df in training_windows.values())
mean_cycles = np.mean([len(find_cycles(df['PLETH'].values)) for df in training_windows.values()])
print(f'Cycles per window — min: {min_cycles},  mean: {mean_cycles:.1f}')

## 2. Synthetic Gap Creation

For each window, `MAX_HOLES_PER_WINDOW` random cardiac cycles are selected. A gap is punched into each selected cycle by replacing the peak (systolic or diastolic) with the interpolated mean of its neighbours — mimicking the flat-line artefact seen in real corrupted recordings.

Two gap-size distributions are created:
- **Uniform** — gap sizes drawn uniformly from 2–100 samples
- **Realistic** — gap sizes drawn from the empirical distribution of real holes

In [ ]:
def punch_hole(window_df: pd.DataFrame, cycle: tuple,
               hole_size: int, cycle_type: str = 'max') -> pd.DataFrame:
    """
    Replace a range of PLETH samples around a cycle peak with a flat mean value.

    Parameters
    ----------
    window_df   : DataFrame containing the PLETH column
    cycle       : (min_start, max_peak, min_end) indices
    hole_size   : number of samples to flatten
    cycle_type  : 'max' to target the systolic peak, 'min' to target a trough

    Returns a modified copy of *window_df*.
    """
    df = window_df.copy()
    pleth = df['PLETH'].copy()
    min_start, max_peak, min_end = cycle

    if cycle_type == 'max':
        sorted_indices = pleth.iloc[min_start:min_end].nlargest(hole_size + 1).index
        target_indices = sorted_indices[:-1]            # top hole_size values
        neighbour_val  = pleth.loc[sorted_indices[-1]]
        edge_val       = pleth.loc[target_indices[-1]]
    else:
        sorted_indices = pleth.iloc[min_start:min_end].nsmallest(hole_size + 1).index
        target_indices = sorted_indices[:-1]
        neighbour_val  = pleth.loc[sorted_indices[-1]]
        edge_val       = pleth.loc[target_indices[-1]]

    fill_value = np.round((neighbour_val + edge_val) / 2)
    start_idx  = target_indices.min()
    end_idx    = target_indices.max()
    pleth.loc[start_idx:end_idx] = fill_value
    df['PLETH'] = pleth
    return df, int(start_idx), int(end_idx)


def create_synthetic_holes(windows: dict, gap_distribution: str = 'uniform') -> tuple:
    """
    Create synthetic gaps in all training windows.

    Parameters
    ----------
    windows          : dict of {window_id: DataFrame}
    gap_distribution : 'uniform' or 'realistic'

    Returns
    -------
    holey_windows : dict of {new_key: DataFrame} with flat-region gaps
    metadata_df   : DataFrame recording gap locations and sizes for scoring
    """
    holey_windows = {}
    records = []

    for wid, df in windows.items():
        pleth = df['PLETH'].values
        cycles = find_cycles(pleth)
        if len(cycles) < MAX_HOLES_PER_WINDOW:
            continue

        selected_cycles = random.sample(cycles, MAX_HOLES_PER_WINDOW)

        for n_holes in range(1, MAX_HOLES_PER_WINDOW + 1):
            record = {'original_window': wid, 'cycles_amount': n_holes}
            modified_df = df.copy()

            for i, cycle in enumerate(selected_cycles[:n_holes], start=1):
                if gap_distribution == 'uniform':
                    hole_size = random.randint(UNIFORM_MIN_GAP, UNIFORM_MAX_GAP)
                else:
                    # Realistic: smaller gaps are more common
                    weights = np.exp(-0.04 * np.arange(UNIFORM_MIN_GAP, UNIFORM_MAX_GAP + 1))
                    hole_size = int(np.random.choice(
                        np.arange(UNIFORM_MIN_GAP, UNIFORM_MAX_GAP + 1),
                        p=weights / weights.sum()
                    ))

                cycle_type = random.choice(['max', 'min'])
                modified_df, start_idx, end_idx = punch_hole(modified_df, cycle, hole_size, cycle_type)

                record[f'hole_size_{i}']  = hole_size
                record[f'type_{i}']       = cycle_type
                record[f'start_idx_{i}']  = start_idx
                record[f'end_idx_{i}']    = end_idx

            new_key = f'{wid}_holes_{n_holes}_{gap_distribution}'
            record['hole_size'] = record.get('hole_size_1', np.nan)  # primary hole size
            holey_windows[new_key] = modified_df
            records.append(record)

    return holey_windows, pd.DataFrame(records)


holey_uniform,   meta_uniform   = create_synthetic_holes(training_windows, 'uniform')
holey_realistic, meta_realistic = create_synthetic_holes(training_windows, 'realistic')

print(f'Uniform distribution   : {len(holey_uniform):>6,} holey windows')
print(f'Realistic distribution : {len(holey_realistic):>6,} holey windows')

## 3. Convert Flat Regions to NaN

Before interpolation, the flat-filled regions are replaced with `NaN` so the interpolation functions know which samples to reconstruct.

In [ ]:
def flat_regions_to_nan(holey_windows: dict, metadata: pd.DataFrame) -> dict:
    """
    Replace the artificially flattened PLETH regions with NaN.

    Uses the start/end indices recorded in *metadata* so that the
    interpolation functions receive a standard gap representation.
    """
    nan_windows = {}
    for _, row in metadata.iterrows():
        n_holes = int(row['cycles_amount'])
        key = f"{int(row['original_window'])}_holes_{n_holes}_{row.get('distribution', 'uniform')}"
        if key not in holey_windows:
            continue
        df = holey_windows[key].copy()
        for i in range(1, n_holes + 1):
            s, e = row.get(f'start_idx_{i}'), row.get(f'end_idx_{i}')
            if pd.notna(s) and pd.notna(e):
                df.loc[int(s):int(e), 'PLETH'] = np.nan
        nan_windows[key] = df
    return nan_windows


meta_uniform['distribution']   = 'uniform'
meta_realistic['distribution'] = 'realistic'

nan_uniform   = flat_regions_to_nan(holey_uniform,   meta_uniform)
nan_realistic = flat_regions_to_nan(holey_realistic, meta_realistic)

print(f'NaN windows ready — uniform: {len(nan_uniform)},  realistic: {len(nan_realistic)}')

## 4. Interpolation Methods

### 4a. Cubic Spline
Fits a piecewise cubic polynomial through the known (non-NaN) points and evaluates it at the gap positions.
Works well for longer gaps (> 100 ms) where the B-spline may over-smooth.

### 4b. B-Spline (k=2)
Fits a quadratic B-spline. More conservative near peaks — better for shorter gaps (< 100 ms).

In [ ]:
def cubic_spline_interpolate(signal_with_nan: np.ndarray) -> np.ndarray:
    """
    Fill NaN values using cubic spline interpolation.

    Parameters
    ----------
    signal_with_nan : 1-D array with NaN at gap positions

    Returns
    -------
    Reconstructed array (no NaN values)
    """
    valid_idx = np.where(~np.isnan(signal_with_nan))[0]
    nan_idx   = np.where(np.isnan(signal_with_nan))[0]
    cs = CubicSpline(valid_idx, signal_with_nan[valid_idx])
    result = signal_with_nan.copy()
    result[nan_idx] = cs(nan_idx)
    return result


def bspline_interpolate(signal_with_nan: np.ndarray, spline_degree: int = 2) -> np.ndarray:
    """
    Fill NaN values using B-spline interpolation.

    Parameters
    ----------
    signal_with_nan : 1-D array with NaN at gap positions
    spline_degree   : degree of the B-spline (default 2 = quadratic)

    Returns
    -------
    Reconstructed array (no NaN values)
    """
    valid_idx = np.where(~np.isnan(signal_with_nan))[0]
    nan_idx   = np.where(np.isnan(signal_with_nan))[0]
    tck = splrep(valid_idx, signal_with_nan[valid_idx], k=spline_degree)
    result = signal_with_nan.copy()
    result[nan_idx] = splev(nan_idx, tck)
    return result


def apply_interpolation(nan_windows: dict, method: str) -> dict:
    """
    Apply *method* interpolation to every window in *nan_windows*.

    Parameters
    ----------
    nan_windows : dict of {key: DataFrame} with NaN in the PLETH column
    method      : 'cubic' or 'bspline'

    Returns
    -------
    dict of {key: DataFrame} with NaN filled
    """
    fn = cubic_spline_interpolate if method == 'cubic' else bspline_interpolate
    result = {}
    for key, df in nan_windows.items():
        imputed_df = df.copy()
        imputed_df['PLETH'] = fn(df['PLETH'].values)
        result[key] = imputed_df
    return result


cubic_uniform   = apply_interpolation(nan_uniform,   'cubic')
cubic_realistic = apply_interpolation(nan_realistic, 'cubic')
bspline_uniform   = apply_interpolation(nan_uniform,   'bspline')
bspline_realistic = apply_interpolation(nan_realistic, 'bspline')

print('Interpolation complete for all window sets.')

## 5. Evaluation — RMSE & Imputation Score

The **Imputation Score** measures how much of the gap's signal error was recovered:

```
Imputation Score = 1 - (RMSE_imputed / RMSE_flat)
```

- Score = 1.0 → perfect reconstruction (imputed = original)
- Score = 0.0 → no improvement over leaving the flat value
- Score < 0.0 → interpolation made things worse

In [ ]:
def compute_imputation_scores(metadata: pd.DataFrame,
                               original_windows: dict,
                               holey_windows: dict,
                               cubic_windows: dict,
                               bspline_windows: dict) -> pd.DataFrame:
    """
    Compute RMSE and Imputation Score for each gap, for both interpolation methods.

    Adds columns to *metadata* in-place and returns the updated DataFrame.
    """
    meta = metadata.copy()
    dist = meta['distribution'].iloc[0] if 'distribution' in meta.columns else 'uniform'

    for idx, row in meta.iterrows():
        wid     = int(row['original_window'])
        n_holes = int(row['cycles_amount'])
        key     = f"{wid}_holes_{n_holes}_{dist}"

        if key not in cubic_windows or wid not in original_windows:
            continue

        original_pleth = original_windows[wid]['PLETH'].values

        for i in range(1, n_holes + 1):
            s = row.get(f'start_idx_{i}')
            e = row.get(f'end_idx_{i}')
            if pd.isna(s) or pd.isna(e):
                continue
            s, e = int(s), int(e)
            indices = slice(s, e + 1)

            orig   = original_pleth[indices]
            flat   = holey_windows[key]['PLETH'].values[indices]
            cubic  = cubic_windows[key]['PLETH'].values[indices]
            bspl   = bspline_windows[key]['PLETH'].values[indices]

            rmse_flat   = np.sqrt(mean_squared_error(orig, flat))
            rmse_cubic  = np.sqrt(mean_squared_error(orig, cubic))
            rmse_bspline= np.sqrt(mean_squared_error(orig, bspl))

            meta.at[idx, f'rmse_flat_{i}']       = rmse_flat
            meta.at[idx, f'rmse_cubic_{i}']      = rmse_cubic
            meta.at[idx, f'rmse_bspline_{i}']    = rmse_bspline
            meta.at[idx, f'score_cubic_{i}']     = 1 - rmse_cubic  / rmse_flat if rmse_flat else np.nan
            meta.at[idx, f'score_bspline_{i}']   = 1 - rmse_bspline / rmse_flat if rmse_flat else np.nan

    return meta


meta_uniform = compute_imputation_scores(
    meta_uniform, training_windows,
    holey_uniform, cubic_uniform, bspline_uniform
)
meta_realistic = compute_imputation_scores(
    meta_realistic, training_windows,
    holey_realistic, cubic_realistic, bspline_realistic
)

# Overall averages
for method in ('cubic', 'bspline'):
    score_cols = [c for c in meta_uniform.columns if c.startswith(f'score_{method}')]
    all_scores = meta_uniform[score_cols].values.flatten()
    valid = all_scores[~np.isnan(all_scores)]
    print(f'{method:10s} — mean imputation score (uniform dist): {valid.mean():.3f}')

## 6. Visualise Results

In [ ]:
# ── Score vs gap size ───────────────────────────────────────────────────────────
holes_range = range(UNIFORM_MIN_GAP, UNIFORM_MAX_GAP + 1)
cubic_mean_by_size   = []
bspline_mean_by_size = []

for hole_size in holes_range:
    cubic_scores, bspline_scores = [], []
    for i in range(1, MAX_HOLES_PER_WINDOW + 1):
        mask = meta_uniform[f'hole_size_{i}'] == hole_size
        cubic_scores.extend(meta_uniform.loc[mask, f'score_cubic_{i}'].dropna())
        bspline_scores.extend(meta_uniform.loc[mask, f'score_bspline_{i}'].dropna())
    cubic_mean_by_size.append(np.mean(cubic_scores) if cubic_scores else np.nan)
    bspline_mean_by_size.append(np.mean(bspline_scores) if bspline_scores else np.nan)

fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(list(holes_range), cubic_mean_by_size,   label='Cubic Spline',  color='steelblue')
ax.plot(list(holes_range), bspline_mean_by_size, label='B-Spline (k=2)', color='orange')
ax.axhline(0, color='grey', linestyle='--', linewidth=0.8)
ax.set_title('Mean Imputation Score by Gap Size (uniform distribution)')
ax.set_xlabel('Gap size (samples)')
ax.set_ylabel('Imputation Score  [1 = perfect,  0 = no gain]')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# ── Side-by-side visual for a random window ────────────────────────────────────
sample_key = random.choice(list(nan_uniform.keys()))
original_id = int(sample_key.split('_')[0])

orig   = training_windows[original_id]['PLETH'].values
holey  = nan_uniform[sample_key]['PLETH'].values
cubic  = cubic_uniform[sample_key]['PLETH'].values
bspl   = bspline_uniform[sample_key]['PLETH'].values

fig, axes = plt.subplots(1, 3, figsize=(18, 3), sharey=True)
for ax, signal, title, color in zip(
    axes,
    [orig, cubic, bspl],
    ['Original', 'Cubic Spline reconstruction', 'B-Spline reconstruction'],
    ['black', 'steelblue', 'orange']
):
    ax.plot(orig,   color='lightgrey', linewidth=1, label='Original')
    ax.plot(signal, color=color,       linewidth=1.2, label=title)
    gap_mask = np.isnan(holey)
    ax.fill_between(range(len(gap_mask)), orig.min(), orig.max(),
                    where=gap_mask, alpha=0.15, color='red', label='Gap region')
    ax.set_title(title)
    ax.set_xlabel('Sample')
axes[0].set_ylabel('PLETH')
axes[2].legend(fontsize=7, loc='upper right')
plt.tight_layout()
plt.show()

## 7. Save Results

In [ ]:
results = {
    'holey_uniform':      holey_uniform,
    'holey_realistic':    holey_realistic,
    'nan_uniform':        nan_uniform,
    'nan_realistic':      nan_realistic,
    'cubic_uniform':      cubic_uniform,
    'cubic_realistic':    cubic_realistic,
    'bspline_uniform':    bspline_uniform,
    'bspline_realistic':  bspline_realistic,
    'meta_uniform':       meta_uniform,
    'meta_realistic':     meta_realistic,
}

with open(PROCESSED_DIR / 'interpolation_results.pkl', 'wb') as f:
    pickle.dump(results, f)

print('Saved → data/processed/interpolation_results.pkl')